# 05 — Least-cost corridors, northern BC + Yukon (**v2**)

Routed corridors connecting the northern proposed IPCAs and existing protected areas, over the
**published O'Brien/Pither transboundary movement-cost surface** at its native **300 m**.

This is the v2 rebuild (decisions **D1–D10**, `docs/05_methods_v2.md`). v1 — a weighted blend of
three regional products, a relative corridor band, a bare MST and a uniform-noise jitter ensemble —
is frozen at git tag `05-v1`, archived in `output_data/corridors_north/_v1_frozen/`, and its
notebook is `archive/05_corridors_north_v1.ipynb`.

**What this claims:** structural-connectivity hypotheses — a robust core versus a flexible
periphery, and which links have no viable alternative. **What it does not claim:** species movement
predictions. Nothing here is validated against movement, genetic or occurrence data.

Run top to bottom. Gate cells assert; if one fails, stop rather than reading past it.

## 1 · Setup

In [ ]:
import importlib
import config, corridors_prep as cp, corridor_graph as cg, corridors_core as cc, corridors_ensemble as ce
for m in (config, cp, cg, cc, ce):
    importlib.reload(m)

KEY = "north"
config.CORRIDORS[KEY]

## 2 · Prep — warp the cost surface to the 300 m routing grid  ·  **gate G2**

One-off. Reprojects EPSG:3347 → ESRI:102008 at native 300 m with `-r near`, so the four ordinal
cost classes `{1, 10, 100, 1000}` survive exactly — at native resolution there is no averaging rule
to defend. Skip once `input_data/corridors_300m/movement_cost.tif` exists.

In [ ]:
g = cp.grid(KEY)
cp.warp(g)
cp.check(g)      # G2: CRS/shape/extent/classes

## 3 · Graph-layer self-test  ·  **gates G4 (in miniature) + adjacency handling**

`corridor_graph` is the only genuinely unit-testable part of the engine, and this repo has no test
runner — so its assertions run here. Covers MST size, that augmentation never perturbs the
backbone, the β ceiling, adjacency exclusion, and quotient-graph centrality.

In [ ]:
cg.selftest()

## 4 · Engine equivalence on the OLD resistance  ·  **gate G1** — the one that matters

Everything else in v2 changes the answer on purpose, so "re-run and expect the same corridors" is
unavailable. Instead the new pipeline is fed **v1's own frozen resistance raster on v1's own 1 km
grid**, restricted to MST-only edges and the relative band. It must reproduce v1's corridor.

If this passes, the refactor is behaviour-preserving and any later v1↔v2 difference is attributable
to D1/D6/D7 rather than to a bug. It also catches the `mcp.traceback` trap — tracebacks read
whichever `find_costs` ran last, so a mis-grouped optimisation silently returns paths from the
wrong source node.

In [ ]:
g1 = cc.gate_g1(KEY)

## 5 · Load the run

Creates `output_data/corridors_north/v2_runNNN/`, writes `run_config.json` (resolved parameters +
git SHA + input hashes — `output_data/` is gitignored, so the run dir is the only provenance that
survives), then builds the 300 m grid cropped to the anchors plus a routing buffer.

`require_cutoff=False` only until §7 has calibrated the band.

In [ ]:
A = cc.start(KEY, label="v2 baseline", require_cutoff=False)
A

### gate G0 — node identity

The node set must survive the 300 m switch. `node_min_km2` replaces v1's resolution-dependent
`node_min_cells=25`, which at 300 m would have meant 2.25 km² and silently admitted different
nodes.

In [ ]:
import json
v1 = json.loads((config.RESULTS_DIR/"corridors_north"/"_v1_frozen"/"corridor_summary.json").read_text())
got = (len(A.nodes), sum(k=="ipca" for k in A.kinds), sum(k=="pa" for k in A.kinds))
exp = (v1["n_nodes"], v1["n_ipca"], v1["n_existing_pa"])
print(f"nodes v2 {got}  vs  v1 {exp}")
assert got == exp, "G0 FAILED: the node set changed; every v1<->v2 comparison would be invalid"
print("G0 OK")

## 6 · Resistance  ·  Phase 1.3 diagnostics

No blend, no free parameters — the surface is used as published. The **effective spread** is the headline: it bounds how much the routing can discriminate at all, which is the standing worry about least-cost modelling over intact northern landscape.

In [ ]:
cc.resistance(A)
cc.resistance_report(A, v1_path=config.RESULTS_DIR/"corridors_north"/"_v1_frozen"/"resistance.tif")

## 7 · Cost-weighted distance  ·  the expensive stage

One MCP pass per node, cached to disk as float32 memmaps keyed by resistance identity. Every
ensemble member below reuses exactly these fields, so this runs **once**.

In [ ]:
cc.cost_distances(A)

### Calibrate the absolute band cutoff (D6) — run once, then hard-code

Bisects for the cutoff whose **MST-only** corridor area reproduces v1's 18,188 km², using the
identical `& ~node_union` area definition, so v1↔v2 route comparisons are not confounded by band
size. Augmentation adds area on top and is reported separately — calibrating against the augmented
network would let the cutoff quietly absorb the augmentation and conflate D6 with D7.

**Write the printed value into `config.CORRIDORS["north"]["cwd_cutoff_abs"]`**, then re-run from §5
without `require_cutoff=False`.

In [ ]:
cutoff, area = cc.calibrate_cutoff(A)

## 8 · Network — MST + bridge-backup augmentation (D7)  ·  **gate G3**

The originally drafted criterion (keep any direct edge with cost ≤ α × MST-path cost) was vacuous:
least-cost distance obeys the triangle inequality, so it admits the complete graph. Replaced by
sequential bridge backup under a cost-ratio ceiling **β** — every added edge is justified by the
named failure it insures, and where nothing clears the ceiling the link is flagged
**irreplaceable**. Those flags are the headline output.

G3 cross-checks the raster flood fill against the graph's own component count.

In [ ]:
cc.corridor_network(A, cutoff=cutoff)

### Criticality table

The funder-facing redundancy justification. `irreplaceable` = no alternative exists at any cost within β; adjacency edges are links between areas that already touch, so they carry no corridor land and cannot be severed by land-use change.

In [ ]:
cols = ["label_i","label_j","cost","in_mst","is_adjacency","ecfb_raw",
        "disconnects","n_pairs_lost","cost_inflation","backup_ratio","irreplaceable","band_km2"]
A.edges[cols].sort_values(["irreplaceable","n_pairs_lost"], ascending=False).head(20)

## 9 · Linkage priority surface (D9)

The primary deliverable: `priority = max_e(edge centrality × band quality)`, graded rather than hard lines — lines invite site-level readings this model cannot support. `max`, not `sum`, so overlap regions are not inflated purely because edges are redundant there.

In [ ]:
cc.priority_surface(A)

In [ ]:
cc.map(A)

## 10 · Compare against v1

Reads v1's frozen `corridors.tif` off disk and reprojects it onto this grid. Routes are *expected* to move — D1 replaced the resistance surface.

In [ ]:
j = cc.compare(A, config.RESULTS_DIR/"corridors_north"/"_v1_frozen",
                label_a="v2 (O'Brien cost, 300 m)", label_b="v1 (blend, 1 km)")

## 11 · Co-benefit audit  ·  **gate G5**

Segments are the connected components of `corridor & ~nodes` — removing node polygons cuts the
network at every PA/IPCA, so the components *are* the physical links.

**The audit runs on the 1 km grid**, not 300 m. Every value layer is natively 1 km; profiling a
300 m mask would sum ~11 replicated cells per source cell against a 1 km denominator and inflate
every "% of Y2Y" figure ~11× while looking entirely plausible.

Read it as an **audit, not a scorecard** — corridors are routed for permeability, never for value,
so a low axis is a finding rather than a failure.

In [ ]:
cc.corridor_profile(A, n_groups=10)

In [ ]:
cc.corridor_group_map(A)

### gate G5 — audit invariance

The "proposed IPCAs" and "existing PAs" rows do not depend on the corridor at all, so they must
reproduce v1's values. That equivalence-checks the whole `_profile_stacks` → `results_core` path —
49 stacks, `reproject_match`, full-Y2Y denominators — in one comparison, and is the primary proof
the 300 m / 1 km grid split is sound.

In [ ]:
import pandas as pd, numpy as np
old = pd.read_csv(config.RESULTS_DIR/"corridors_north"/"_v1_frozen"/"corridor_profile.csv")
new = A.profile["table"]
for area in ("proposed IPCAs", "existing PAs"):
    o = old[old.area == area].iloc[0]; n = new[new.area == area].iloc[0]
    cols = [c for c in new.columns if "richness" in c and c in old.columns]
    d = max(abs(float(o[c]) - float(n[c])) for c in cols)
    print(f"  {area:16s} max |Δrichness| over {len(cols)} axes = {d:.4f}")
    assert d < 0.02, f"G5 FAILED on {area}: the audit path changed (max Δ {d:.4f})"
print("G5 OK — the audit is unchanged by the routing-grid switch")

## 12 · Structured ensemble (D8)  ·  **gate G7**

Replaces the jitter ensemble. Three axes, each mapped to a documented assumption:

| axis | question |
|---|---|
| **B** band cutoff | how wide is a corridor? |
| **C** node leave-one-out | what is contingent on a single unrealized proposal? |
| **D** β sweep | what counts as a viable alternative? |

Axis **A** (component cost perturbation) is deferred behind **H2**.

All three reuse the cached CWD, so this is graph rebuilds rather than re-routing. **Axis C is the
one to read closely** — Dene Kʼéh Kusān wraps 11 BC parks and carries roughly a quarter of the
network's edges as adjacencies, and it is a *proposal, not yet realized*.

In [ ]:
ce.gate_g7(A)

In [ ]:
ce.run(A)

In [ ]:
ce.collect(A)

## 13 · Write outputs

In [ ]:
cc.finish(A)

In [ ]:
cc.runs(KEY)